In [5]:
import numpy as np

# 1. 원본 평점 행렬 R 정의 (세 번째 행 끝에 쉼표 추가)
R = np.array([[4, np.nan, np.nan, 2, np.nan],
              [np.nan, 5, np.nan, 3, 1],
              [np.nan, np.nan, 3, 4, 4], # 여기에 쉼표가 있어야 합니다.
              [5, 2, 1, 2, np.nan]])

num_users, num_items = R.shape
K = 3 # 잠재 요인 수

np.random.seed(1)

# 2. P 행렬: (사용자 수, K)
P = np.random.normal(scale=1./K, size=(num_users, K))

# 3. Q 행렬: (아이템 수, K)로 수정 (num_users -> num_items)
# 나중에 예측 평점을 구할 때 P와 Q의 전치행렬(Q.T)을 곱하게 됩니다.
Q = np.random.normal(scale=1./K, size=(num_items, K))


In [6]:
import numpy as np
from sklearn.metrics import mean_squared_error

# 1. RMSE를 계산하는 함수 정의
# 실제 평점이 있는 요소에 대해서만 오차를 계산합니다.
def get_rmse(R, P, Q, non_zeros):
    error = 0
    # 예측된 평점 행렬 계산 (P와 Q의 전치행렬 곱)
    full_pred_matrix = np.dot(P, Q.T)

    # 실제 평점 행렬에서 0이 아닌(평점이 매겨진) 위치의 인덱스 추출
    x_non_zero_ind = [non_zero[0] for non_zero in non_zeros]
    y_non_zero_ind = [non_zero[1] for non_zero in non_zeros]
    R_non_zeros = R[x_non_zero_ind, y_non_zero_ind]

    # 예측 행렬에서도 동일한 위치의 값만 추출
    full_pred_matrix_non_zeros = full_pred_matrix[x_non_zero_ind, y_non_zero_ind]

    # RMSE 계산
    mse = mean_squared_error(R_non_zeros, full_pred_matrix_non_zeros)
    rmse = np.sqrt(mse)

    return rmse

# 2. 행렬 분해 메인 함수 정의 (SGD 방식)
def matrix_factorization(R, K, steps=200, learning_rate=0.01, r_lambda = 0.01):
    num_users, num_items = R.shape

    # P와 Q 매트릭스를 정규 분포를 가진 랜덤한 값으로 초기화
    np.random.seed(1)
    P = np.random.normal(scale=1./K, size=(num_users, K))
    Q = np.random.normal(scale=1./K, size=(num_items, K))

    # 실제 평점이 있는(0보다 큰) 위치와 값을 리스트에 저장
    non_zeros = [ (i, j, R[i,j]) for i in range(num_users) for j in range(num_items) if R[i,j] > 0 ]

    # SGD 기법을 이용해 P와 Q를 반복적으로 업데이트
    for step in range(steps):
        for i, j, r in non_zeros:
            # 실제 평점과 예측 평점의 차이(오차) 계산
            eij = r - np.dot(P[i, :], Q[j, :].T)

            # 규제(Regularization)를 포함한 업데이트 공식 적용
            P[i,:] = P[i,:] + learning_rate*(eij * Q[j,:] - r_lambda*P[i,:])
            Q[j,:] = Q[j,:] + learning_rate*(eij * P[i,:] - r_lambda*Q[j,:])

        # 10회 반복마다 RMSE 출력
        rmse = get_rmse(R, P, Q, non_zeros)
        if (step % 10) == 0 :
            print("### iteration step : ", step ," rmse : ", rmse)

    return P, Q

# 3. 실험용 데이터 설정 및 실행
# 원본 평점 행렬 R (결측치는 0으로 표시) [cite: 59, 198]
R = np.array([[4, 0, 0, 2, 0],
              [0, 5, 0, 3, 1],
              [0, 0, 3, 4, 4],
              [5, 2, 1, 2, 0]])

# 행렬 분해 수행 (잠재 요인 K=3)
P, Q = matrix_factorization(R, K=3, steps=200, learning_rate=0.01, r_lambda = 0.01)

# 최종 예측 행렬 결과 확인
pred_matrix = np.dot(P, Q.T)
print("\n##### 예측 행렬 결과 #####")
print(np.round(pred_matrix, 2))

### iteration step :  0  rmse :  3.2388050277987723
### iteration step :  10  rmse :  2.916635469036195
### iteration step :  20  rmse :  2.198706863150832
### iteration step :  30  rmse :  1.3045525302788117
### iteration step :  40  rmse :  0.7565127655009479
### iteration step :  50  rmse :  0.4876723101369648
### iteration step :  60  rmse :  0.3562311584670397
### iteration step :  70  rmse :  0.27952329795334957
### iteration step :  80  rmse :  0.22656292990066265
### iteration step :  90  rmse :  0.18700753399789358
### iteration step :  100  rmse :  0.1564340384819247
### iteration step :  110  rmse :  0.13234649522249217
### iteration step :  120  rmse :  0.11311333721232043
### iteration step :  130  rmse :  0.09759058709002262
### iteration step :  140  rmse :  0.0849441568587214
### iteration step :  150  rmse :  0.07455141311978046
### iteration step :  160  rmse :  0.06594094420477092
### iteration step :  170  rmse :  0.05875268710429461
### iteration step :  180  rmse 

In [8]:
pip install scikit-surprise

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.4/154.4 kB 8.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for scikit-surprise: filename=scikit_surprise-1.1.4-cp312-cp312-linux_x86_64.whl size=2555930 sha256=85df86b3f4d812156c8d65262fa8e78b8b383934888a93c9c6979cec4fcc74b8
  Stored in directory: /root/.cache/pip/wheels/75/fa/bc/739bc2cb1fbaab6061854e6cfbb81a0ae52c92a502a7fa454b
Successfully built scikit-surprise


In [1]:
from surprise import SVD
from surprise import Dataset
from surprise import accuracy
from surprise.model_selection import train_test_split

In [10]:
!pip install 'numpy<2'

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 57.4 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
shap 0.50.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
pytensor 2.35.1 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jax 

In [3]:
data = Dataset.load_builtin('ml-100k')
# 수행     시마다    동일하게    데이터를    분할하기     위해    random.state 값    부여
trainset, testset = train_test_split(data,  test_size=.25, random_state=0)

Dataset ml-100k could not be found. Do you want to download it? [Y/n] Y
Trying to download dataset from https://files.grouplens.org/datasets/movielens/ml-100k.zip...
Done! Dataset ml-100k has been saved to /root/.surprise_data/ml-100k


In [4]:
algo = SVD(random_state=0)
algo.fit(trainset)

In [5]:
predictions = algo.test( testset )
print('prediction type:', type(predictions), 'size：',len(predictions))
print('prediction 결과의     최초    5개     추출')
predictions[:5]

prediction type: <class 'list'> size： 25000
prediction 결과의     최초    5개     추출


[Prediction(uid='120', iid='282', r_ui=4.0, est=3.5114147666251547, details={'was_impossible': False}),
 Prediction(uid='882', iid='291', r_ui=4.0, est=3.573872419581491, details={'was_impossible': False}),
 Prediction(uid='535', iid='507', r_ui=5.0, est=4.033583485472447, details={'was_impossible': False}),
 Prediction(uid='697', iid='244', r_ui=5.0, est=3.8463639495936905, details={'was_impossible': False}),
 Prediction(uid='751', iid='385', r_ui=4.0, est=3.1807542478219157, details={'was_impossible': False})]

In [6]:
[    (pred.uid, pred.iid, pred.est) for pred in predictions[:3]  ]

[('120', '282', 3.5114147666251547),
 ('882', '291', 3.573872419581491),
 ('535', '507', 4.033583485472447)]

In [7]:
uid = str(196)
iid = str(302)
pred = algo.predict(uid,iid)
print(pred)

user: 196        item: 302        r_ui = None   est = 4.49   {'was_impossible': False}


In [8]:
accuracy.rmse(predictions)

RMSE: 0.9467


0.9466860806937948

In [13]:
import pandas as pd
from surprise import Dataset

# Surprise가 다운로드한 ml-100k 데이터셋의 실제 경로와 파일 이름 (u.data)을 사용합니다.
# ml-100k 데이터셋의 u.data 파일은 사용자ID, 아이템ID, 평점, 타임스탬프 순서로 구성되어 있으며 탭으로 구분됩니다.
# Surprise의 Dataset.load_builtin('ml-100k')는 이미 이 데이터를 로드하여 내부적으로 사용하고 있습니다.
# 만약 pandas를 사용하여 이 데이터를 로드하려면 다음처럼 할 수 있습니다.

# ml-100k 데이터셋의 ratings 파일은 'u.data'입니다.
# 이 파일은 사용자 ID, 영화 ID, 평점, 타임스탬프로 구성되어 있으며, 탭으로 구분됩니다.
ratings_file_path = '/root/.surprise_data/ml-100k/ml-100k/u.data' # './' 제거하여 절대 경로로 수정

# pandas로 u.data 파일을 읽을 때, 컬럼 이름과 구분자(sep='\t')를 명시해야 합니다.
ratings_df = pd.read_csv(ratings_file_path, sep='\t', header=None,
                       names=['user_id', 'item_id', 'rating', 'timestamp'])

# 데이터프레임의 처음 5행을 출력하여 확인합니다.
print(ratings_df.head())

# 필요한 경우, Surprise에서 요구하는 형식 (user, item, rating)으로 저장할 수 있습니다.
# 예를 들어, 'ratings_noh.csv' 파일로 저장하려면:
# ratings_df[['user_id', 'item_id', 'rating']].to_csv(
#    './root/.surprise_data/ml-100k/ml-100k/ratings_noh.csv', index=False, header=False
# )

   user_id  item_id  rating  timestamp
0      196      242       3  881250949
1      186      302       3  891717742
2       22      377       1  878887116
3      244       51       2  880606923
4      166      346       1  886397596


In [11]:
!ls -R /root/.surprise_data/ml-100k

/root/.surprise_data/ml-100k:
ml-100k

/root/.surprise_data/ml-100k/ml-100k:
allbut.pl  u1.base  u2.test  u4.base  u5.test  ub.base	u.genre  u.occupation
mku.sh	   u1.test  u3.base  u4.test  ua.base  ub.test	u.info	 u.user
README	   u2.base  u3.test  u5.base  ua.test  u.data	u.item


In [16]:
from surprise import Reader
reader = Reader(line_format='user item rating timestamp',  sep='\t',rating_scale=(0.5, 5))
data=Dataset.load_from_file('/root/.surprise_data/ml-100k/ml-100k/u.data',  reader=reader)

In [19]:
# 1. 데이터셋을 trainset과 testset으로 분할
# train_test_split은 Trainset 객체와 raw testset (list of tuples)을 반환합니다.
trainset, testset = train_test_split(data, test_size=0.25, random_state=0)

# 2. SVD 모델 학습
algo = SVD(n_factors=50, random_state=0)
algo.fit(trainset) # trainset은 이미 surprise.Trainset 객체입니다.

# 3. 예측은 testset을 사용하여 수행합니다.
predictions = algo.test(testset) # testset은 raw ratings의 list입니다.
accuracy.rmse(predictions)

RMSE: 0.9458


0.9457855197571977

In [21]:
import pandas as pd
from surprise import Reader, Dataset
ratings = pd.read_csv('/root/.surprise_data/ml-100k/ml-100k/u.data', sep='\t', header=None, names=['userid', 'movield', 'rating', 'timestamp'])
reader = Reader(rating_scale=(0.5,  5.0))
#   ratings DataFrame에서     칼럼은    사용자    아이디, 아이템     아이디, 평점     순서를    지켜야    합니다.
data = Dataset.load_from_df(ratings[['userid', 'movield',   'rating']],  reader)
trainset, testset = train_test_split(data,   test_size=.25, random_state=0)
algo = SVD(n_factors=50, random_state=0)
algo.fit(trainset)
predictions = algo.test( testset )
accuracy.rmse(predictions)

RMSE: 0.9458


0.9457855197571977

In [23]:
from surprise.model_selection import cross_validate
#    판다스   DataFrame에서     Surprise 데이터     세트로    데이터     로딩
reader = Reader(rating_scale=(0.5,  5.0))
data = Dataset.load_from_df(ratings[['userid',  'movield',   'rating']],  reader)
algo = SVD(random_state=0)
cross_validate(algo, data, measures=[ 'RMSE',   'MAE'], cv=5,   verbose=True)

Evaluating RMSE, MAE of algorithm SVD on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.9409  0.9388  0.9277  0.9483  0.9331  0.9378  0.0070  
MAE (testset)     0.7420  0.7416  0.7293  0.7451  0.7367  0.7389  0.0055  
Fit time          1.32    1.31    1.37    1.94    1.32    1.45    0.25    
Test time         0.12    0.25    0.18    0.10    0.27    0.18    0.07    


{'test_rmse': array([0.94093683, 0.93876325, 0.92773331, 0.94829749, 0.93311353]),
 'test_mae': array([0.74197351, 0.74163748, 0.72931499, 0.74508501, 0.73666611]),
 'fit_time': (1.3230950832366943,
  1.310234546661377,
  1.3650386333465576,
  1.9425837993621826,
  1.321216106414795),
 'test_time': (0.11644506454467773,
  0.2541470527648926,
  0.17620587348937988,
  0.09936308860778809,
  0.2698531150817871)}

In [27]:
from surprise.model_selection import GridSearchCV

param_grid = {'n_epochs': [20,40,60],'n_factors':[50,100,200]}

gs = GridSearchCV(SVD, param_grid, measures=['rmse', 'mse', 'mae'], cv=3)
gs.fit(data)
print(gs.best_score['rmse'])
print(gs.best_params['rmse'])

0.943753389457367
{'n_epochs': 20, 'n_factors': 50}


In [29]:
data = Dataset.load_from_df(ratings[['userid',  'movield',  'rating']],  reader)
# DatasetAutoFolds 객체에서 전체 학습셋을 만듭니다.
trainset = data.build_full_trainset()
algo = SVD(n_factors=50, random_state=0)
algo.fit(trainset)

In [32]:
from surprise.dataset import DatasetAutoFolds
reader = Reader(line_format='user item rating timestamp',  sep='\t', rating_scale=(0.5, 5))

data_folds = DatasetAutoFolds(ratings_file ='/root/.surprise_data/ml-100k/ml-100k/u.data', reader = reader )

trainset = data_folds.build_full_trainset()
algo = SVD(n_epochs=20, n_factors=50, random_state=0)
algo.fit(trainset)

In [34]:
movies = pd.read_csv('/root/.surprise_data/ml-100k/ml-100k/u.item', sep='|', header=None, encoding='latin1')
# 'u.item' 파일은 컬럼이 '|' (파이프)로 구분되어 있고 헤더가 없으므로 이를 명시합니다.
# 또한, 영화 상세 정보 파일에는 'movieid'라는 컬럼 이름이 없으므로, 영화 ID가 첫 번째 컬럼에 있다고 가정하고 적절한 컬럼 이름을 지정하거나 인덱스를 사용하여 접근해야 합니다.
# 여기서는 임시로 첫 번째 컬럼을 'movieid'로 가정하고 진행합니다.
movies.columns = ['movieid', 'title', 'release_date', 'video_release_date', 'imdb_url', 'unknown', 'Action', 'Adventure', 'Animation', "Children's", 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']

# 'ratings' DataFrame에는 'movield' 컬럼이 있고, 'u.item'에서는 'movieid' 컬럼이 사용됩니다.
# 'movield'를 사용하여 해당 영화 ID를 찾아야 합니다.
movieIds = ratings[ratings['userid']==9]['movield']

if movieIds[movieIds == 42].count()==0:
  print('사용자 아이디 9는 영화 아이디 42의 평점 없음')

print(movies[movies['movieid']==42])

사용자 아이디 9는 영화 아이디 42의 평점 없음
    movieid          title release_date  video_release_date  \
41       42  Clerks (1994)  01-Jan-1994                 NaN   

                                            imdb_url  unknown  Action  \
41  http://us.imdb.com/M/title-exact?Clerks%20(1994)        0       0   

    Adventure  Animation  Children's  ...  Fantasy  Film-Noir  Horror  \
41          0          0           0  ...        0          0       0   

    Musical  Mystery  Romance  Sci-Fi  Thriller  War  Western  
41        0        0        0       0         0    0        0  

[1 rows x 24 columns]


In [35]:
uid = str(9)
iid = str(42)
pred = algo.predict(uid, iid, verbose=True)

user: 9          item: 42         r_ui = None   est = 4.25   {'was_impossible': False}


In [36]:
def get_unseen_surprise(ratings, movies, userid):
  seen_movies = ratings[ratings['userid']==userid]['movield'].tolist()
  total_movies = movies['movieid'].tolist()
  unseen_movies = [movie for movie in total_movies if movie not in seen_movies]
  print('평점     매긴     영화    수:',  len(seen_movies),'추천     대상    영화   수:',  len(unseen_movies),
'전체     영화   수:', len(total_movies))
  return unseen_movies
unseen_movies = get_unseen_surprise(ratings, movies, 9)


평점     매긴     영화    수: 22 추천     대상    영화   수: 1660 전체     영화   수: 1682


In [40]:

def get_unseen_surprise(ratings, movies, userId):
    # 특정 사용자가 평점을 매긴 모든 영화 리스트 추출
    seen_movies = ratings[ratings['userid'] == userId]['movield'].tolist()

    # 모든 영화의 movieId 리스트 추출
    total_movies = movies['movieid'].tolist()

    # 모든 영화 중 이미 본 영화를 제외하여 미시청 영화 리스트 생성
    unseen_movies = [movie for movie in total_movies if movie not in seen_movies]

    print(f'### 사용자 {userId}가 시청하지 않은 영화 개수: {len(unseen_movies)}')
    return unseen_movies

def recomm_movie_by_surprise(algo, userId, unseen_movies, movies, top_n=10):
    # 1. 미시청 영화들에 대해 알고리즘 객체의 predict()를 호출하여 예측 평점 계산
    predictions = [algo.predict(str(userId), str(movieId)) for movieId in unseen_movies]

    # 2. 예측 평점(est)을 기준으로 내림차순 정렬하기 위한 내부 함수
    def sortkey_est(pred):
        return pred.est

    # 3. 정렬 후 상위 top_n개 추출
    predictions.sort(key=sortkey_est, reverse=True)
    top_predictions = predictions[:top_n]

    # 4. 상위 영화들의 상세 정보(ID, 예측 평점, 제목) 추출
    top_movie_ids = [int(pred.iid) for pred in top_predictions]
    top_movie_rating = [pred.est for pred in top_predictions]

    # 영화 제목(Title)을 가져오기 위해 movies 데이터프레임과 매칭
    # (movies 데이터프레임의 인덱스 기준이 아닌 movieId 컬럼 기준으로 필터링)
    top_movie_titles = movies[movies.movieid.isin(top_movie_ids)]['title']

    # 최종 리스트 생성: (영화ID, 영화제목, 예측평점)
    top_movie_preds = [ (id, title, rating) for id, title, rating in \
                        zip(top_movie_ids, top_movie_titles, top_movie_rating)]

    return top_movie_preds

unseen_movies = get_unseen_surprise(ratings, movies, 9)
top_movie_preds = recomm_movie_by_surprise(algo, 9, unseen_movies, movies, top_n=10)
print('#### Top-10 추천     영화    리스트    #####')
for top_movie in top_movie_preds:
  print(top_movie[1],";", top_movie[2])

### 사용자 9가 시청하지 않은 영화 개수: 1660
#### Top-10 추천     영화    리스트    #####
Usual Suspects, The (1995) ; 4.998461495674477
Shawshank Redemption, The (1994) ; 4.9581509884065245
Wallace & Gromit: The Best of Aardman Animation (1996) ; 4.919645463559541
Wrong Trousers, The (1993) ; 4.882499471175927
Empire Strikes Back, The (1980) ; 4.87836594561176
Raiders of the Lost Ark (1981) ; 4.873833196467519
L.A. Confidential (1997) ; 4.8584916788987496
One Flew Over the Cuckoo's Nest (1975) ; 4.820999931348582
Close Shave, A (1995) ; 4.795043987320939
North by Northwest (1959) ; 4.789671087078756
